In [ ]:

# ==========================================================
# PROJECT 6 - SMART AGRICULTURE ANALYTICS
# REAL KAGGLE DATA | ONE CELL | GOOGLE COLAB
# ==========================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "kagglehub", "pandas", "numpy", "plotly", "openpyxl"
])

import pandas as pd
import numpy as np
import plotly.express as px
import kagglehub
import zipfile
import re
from pathlib import Path
from plotly.io import to_html
from IPython.display import display

# ----------------------------------------------------------
# 1. SETTINGS
# ----------------------------------------------------------

PROJECT = Path("/content/Smart_Agriculture_Analytics")
PROJECT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ZIP = False

DATASETS = [
    "abhinand05/crop-production-in-india",
    "srinivas1/agricuture-crops-production-in-india"
]

# ----------------------------------------------------------
# 2. DATASET READING FUNCTIONS
# ----------------------------------------------------------

def normalize(value):
    return re.sub(
        r"[^a-z0-9]+", "_", str(value).lower()
    ).strip("_")


def read_file(path):

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:

            try:
                return pd.read_csv(
                    path,
                    encoding=encoding,
                    low_memory=False
                )

            except UnicodeDecodeError:
                continue

    raise ValueError("Unsupported dataset file")


aliases = {
    "State": [
        "state_name", "state", "states"
    ],
    "District": [
        "district_name", "district"
    ],
    "Year": [
        "crop_year", "year", "cropyear"
    ],
    "Season": [
        "season", "crop_season"
    ],
    "Crop": [
        "crop", "crop_name", "crop_type"
    ],
    "Area": [
        "area", "area_ha", "area_hectares",
        "area_in_hectares", "cultivated_area"
    ],
    "Production": [
        "production", "production_tonnes",
        "production_tons", "production_mt",
        "total_production"
    ]
}


def standardize(frame):

    frame = frame.copy()

    frame.columns = [
        normalize(c) for c in frame.columns
    ]

    mapping = {}

    for standard, possible in aliases.items():

        for name in possible:

            if name in frame.columns:
                mapping[name] = standard
                break

    return frame.rename(columns=mapping)


# ----------------------------------------------------------
# 3. DOWNLOAD REAL KAGGLE DATA
# ----------------------------------------------------------

df = None
selected_dataset = None
selected_file = None

for dataset_id in DATASETS:

    try:
        print("\nChecking Kaggle:", dataset_id)

        folder = Path(
            kagglehub.dataset_download(dataset_id)
        )

        candidates = [
            f for f in folder.rglob("*")
            if f.suffix.lower() in [
                ".csv", ".xlsx", ".xls"
            ]
        ]

        for file in candidates:

            try:
                temp = standardize(read_file(file))

                print("File:", file.name)
                print("Columns:", list(temp.columns))

                required = {
                    "Crop", "Area", "Production"
                }

                if (
                    required.issubset(temp.columns)
                    and (
                        "State" in temp.columns
                        or "District" in temp.columns
                    )
                ):
                    df = temp.copy()
                    selected_dataset = dataset_id
                    selected_file = file.name
                    break

            except Exception as error:
                print("Skipped:", file.name, error)

        if df is not None:
            break

    except Exception as error:
        print("Dataset error:", error)


if df is None:
    raise ValueError(
        "Compatible dataset not found. "
        "Please share the printed column names."
    )

print("\nDATASET LOADED SUCCESSFULLY")
print("Source:", selected_dataset)
print("File:", selected_file)
print("Original records:", len(df))

# ----------------------------------------------------------
# 4. DATA CLEANING
# ----------------------------------------------------------

original_rows = len(df)

for col in ["State", "District", "Crop", "Season"]:

    if col in df.columns:

        df[col] = (
            df[col].astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

for col in ["Area", "Production"]:

    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

    df.loc[df[col] < 0, col] = np.nan

if "Year" in df.columns:

    df["Year"] = pd.to_numeric(
        df["Year"].astype(str)
        .str.extract(r"(\d{4})")[0],
        errors="coerce"
    )

df = df.dropna(
    subset=["Crop", "Area", "Production"]
)

df = df[df["Area"] > 0]

df = df.drop_duplicates().reset_index(drop=True)

if df.empty:
    raise ValueError("No usable records after cleaning.")

df["Yield_Ratio"] = (
    df["Production"] / df["Area"]
)

df.to_csv(
    PROJECT / "cleaned_agriculture_data.csv",
    index=False
)

print("\nCLEANING COMPLETED")
print("Original rows:", original_rows)
print("Clean rows:", len(df))

display(df.head())

# ----------------------------------------------------------
# 5. KPI SUMMARY
# ----------------------------------------------------------

kpi = {
    "Total Records": len(df),
    "Total Crops": df["Crop"].nunique(),
    "Total Area": df["Area"].sum(),
    "Total Production": df["Production"].sum()
}

for col in ["State", "District", "Year", "Season"]:

    if col in df.columns:
        kpi["Unique " + col] = df[col].nunique()

kpi_df = pd.DataFrame(
    list(kpi.items()),
    columns=["KPI", "Value"]
)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

print("\nPROJECT KPIs")
display(kpi_df)

# ----------------------------------------------------------
# 6. INTERACTIVE PLOTLY CHARTS
# ----------------------------------------------------------

figures = []


def show_chart(fig, title):

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(
            l=40, r=40, t=80, b=50
        )
    )

    figures.append((title, fig))
    fig.show()


def grouped_sum(column, metric="Production"):

    return (
        df.dropna(subset=[column])
        .groupby(column, as_index=False)[metric]
        .sum()
    )


# CHART 1 - TOP STATES

if "State" in df.columns:

    data = grouped_sum("State")

    data = data.nlargest(
        15, "Production"
    ).sort_values("Production")

    show_chart(
        px.bar(
            data,
            x="Production",
            y="State",
            orientation="h",
            color="Production"
        ),
        "1. Top 15 States by Production"
    )


# CHART 2 - TOP CROPS

data = grouped_sum("Crop")

data = data.nlargest(
    15, "Production"
).sort_values("Production")

show_chart(
    px.bar(
        data,
        x="Production",
        y="Crop",
        orientation="h",
        color="Production"
    ),
    "2. Top 15 Crops by Production"
)


# CHART 3 - YEARLY PRODUCTION

if "Year" in df.columns:

    data = grouped_sum("Year")
    data = data.sort_values("Year")

    show_chart(
        px.line(
            data,
            x="Year",
            y="Production",
            markers=True
        ),
        "3. Yearly Crop Production Trend"
    )


# CHART 4 - SEASONAL PRODUCTION

if "Season" in df.columns:

    data = grouped_sum("Season")

    show_chart(
        px.pie(
            data,
            names="Season",
            values="Production",
            hole=0.4
        ),
        "4. Production by Season"
    )


# CHART 5 - TOP DISTRICTS

if "District" in df.columns:

    data = grouped_sum("District")

    data = data.nlargest(
        15, "Production"
    ).sort_values("Production")

    show_chart(
        px.bar(
            data,
            x="Production",
            y="District",
            orientation="h",
            color="Production"
        ),
        "5. Top 15 Districts by Production"
    )


# CHART 6 - AREA VS PRODUCTION

scatter = df[
    ["Area", "Production", "Crop"]
].dropna()

if len(scatter) > 12000:

    scatter = scatter.sample(
        12000, random_state=42
    )

show_chart(
    px.scatter(
        scatter,
        x="Area",
        y="Production",
        hover_data=["Crop"],
        opacity=0.5
    ),
    "6. Cultivated Area vs Production"
)


# CHART 7 - CROP YIELD RATIO

data = (
    df.groupby("Crop", as_index=False)
    [["Area", "Production"]]
    .sum()
)

data = data[data["Area"] > 0]

data["Yield_Ratio"] = (
    data["Production"] / data["Area"]
)

data = data.nlargest(
    15, "Yield_Ratio"
).sort_values("Yield_Ratio")

show_chart(
    px.bar(
        data,
        x="Yield_Ratio",
        y="Crop",
        orientation="h",
        color="Yield_Ratio"
    ),
    "7. Top 15 Crops by Production-to-Area Ratio"
)


# CHART 8 - STATE VS CROP HEATMAP

if "State" in df.columns:

    states = (
        df.groupby("State")["Production"]
        .sum().nlargest(10).index
    )

    crops = (
        df.groupby("Crop")["Production"]
        .sum().nlargest(10).index
    )

    subset = df[
        df["State"].isin(states)
        & df["Crop"].isin(crops)
    ]

    matrix = subset.pivot_table(
        index="State",
        columns="Crop",
        values="Production",
        aggfunc="sum",
        fill_value=0
    )

    show_chart(
        px.imshow(
            matrix,
            aspect="auto",
            labels={"color": "Production"}
        ),
        "8. State vs Crop Production Heatmap"
    )


# ----------------------------------------------------------
# 7. EXPORT CSV REPORTS
# ----------------------------------------------------------

for column in [
    "State", "District", "Crop",
    "Year", "Season"
]:

    if column not in df.columns:
        continue

    report = (
        df.groupby(column, dropna=False)
        [["Area", "Production"]]
        .sum()
        .reset_index()
    )

    report["Yield_Ratio"] = np.where(
        report["Area"] > 0,
        report["Production"] / report["Area"],
        np.nan
    )

    report.to_csv(
        PROJECT / f"summary_{column.lower()}.csv",
        index=False
    )


# ----------------------------------------------------------
# 8. COMBINED INTERACTIVE DASHBOARD
# ----------------------------------------------------------

html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Smart Agriculture Analytics</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p {
    text-align: center;
}
.chart {
    background: white;
    border-radius: 12px;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
}
</style>
</head>
<body>
<h1>SMART AGRICULTURE ANALYTICS</h1>
<p>Real Kaggle Crop Production Data</p>
"""

for i, (title, fig) in enumerate(figures):

    chart_html = to_html(
        fig,
        full_html=False,
        include_plotlyjs=(
            True if i == 0 else False
        )
    )

    html += (
        '<div class="chart">'
        + chart_html
        + '</div>'
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Smart_Agriculture_Dashboard.html"
)

dashboard_path.write_text(
    html, encoding="utf-8"
)


# ----------------------------------------------------------
# 9. README
# ----------------------------------------------------------

readme = f"""
SMART AGRICULTURE ANALYTICS

Dataset:
https://www.kaggle.com/datasets/{selected_dataset}

Source File:
{selected_file}

Clean Records:
{len(df)}

Interactive Charts:
{len(figures)}

No synthetic records generated.

Note:
This analysis uses crop-production records.
Soil nutrients, rainfall and temperature
are not included unless present in the source.
Verify production and area units.
"""

(PROJECT / "README.txt").write_text(
    readme, encoding="utf-8"
)


# ----------------------------------------------------------
# 10. CREATE ZIP
# ----------------------------------------------------------

zip_path = Path(
    "/content/Smart_Agriculture_Analytics.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for file in PROJECT.rglob("*"):

        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )


# ----------------------------------------------------------
# 11. FINAL OUTPUT
# ----------------------------------------------------------

print("\n" + "=" * 45)
print("PROJECT 6 COMPLETED")
print("=" * 45)

print("Dataset:", selected_dataset)
print("Clean records:", len(df))
print("Interactive charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
